# P01 · Forward and inverse kinematics, Jacobians and singularities

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/09_planning_learning/P01_forward_and_inverse_kinematics_jacobians_and_sin.ipynb)

**Track:** Planning and learning · **Skill axis (sheet):** Planning and learning · **Time:** about 60 min

**Skill Hub concepts:** Forward and inverse kinematics; Jacobian and singularities; Singularity

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("P01")

## Map: joints to end effector and back

Two-link planar arm, link lengths $l_1 = 0.30$ m, $l_2 = 0.25$ m, joint angles $q = (q_1, q_2)$:
$$x = l_1\cos q_1 + l_2\cos(q_1 + q_2), \qquad y = l_1\sin q_1 + l_2\sin(q_1 + q_2)$$
**Jacobian** (maps joint velocities to end-effector velocity, $\dot{p} = J(q)\,\dot q$):
$$J = \begin{bmatrix} -l_1 s_1 - l_2 s_{12} & -l_2 s_{12} \\ l_1 c_1 + l_2 c_{12} & l_2 c_{12} \end{bmatrix}, \qquad \det J = l_1 l_2 \sin q_2$$
**Inverse kinematics** (law of cosines): $\cos q_2 = \dfrac{x^2 + y^2 - l_1^2 - l_2^2}{2 l_1 l_2}$, two solutions
$q_2 = \pm\arccos(\cdot)$ (elbow up and down), then $q_1 = \operatorname{atan2}(y, x) - \operatorname{atan2}(l_2\sin q_2,\, l_1 + l_2\cos q_2)$.

| Situation | $\det J$ | Consequence |
|---|---|---|
| Arm fully stretched or folded ($q_2 = 0$ or $\pi$) | 0 | **singularity**: no motion along the arm direction; $J^{-1}$ blows up |
| Near singularity | small | tiny Cartesian moves demand huge joint speeds |
| Elsewhere | non-zero | locally invertible |

**Damped least squares** (Levenberg-Marquardt IK) stays bounded near singularities:
$$\Delta q = J^\top\left(J J^\top + \lambda^2 I\right)^{-1} e$$
Each step is bounded: $\lVert\Delta q\rVert \le \lVert e\rVert / (2\lambda)$. Too small a $\lambda$ with an unreachable
target makes the arm chatter around the stretched pose; try $\lambda = 0.05$ and $0.1$ in Task 3.
For 6-axis arms the same ideas hold with $6 \times 6$ Jacobians (wrist, shoulder and elbow singularities).

In [ ]:
import numpy as np
L1, L2 = 0.30, 0.25

## Task 1: forward kinematics and the analytic Jacobian

In [ ]:
def fk(q):
    """Return np.array([x, y])."""
    raise NotImplementedError  # TODO

def jacobian(q):
    raise NotImplementedError  # TODO

lab.check("fk", fk, jacobian)

## Task 2: analytic inverse kinematics
Return `(q1, q2)` for the requested elbow (`elbow_up=True` means $q_2 < 0$ here), or `None` if unreachable.

In [ ]:
def ik(x, y, elbow_up=True):
    raise NotImplementedError  # TODO

lab.check("ik", ik, fk)

## Task 3: damped least squares near a singularity
One step: return $q + \Delta q$ with $e = \text{target} - \text{fk}(q)$.

In [ ]:
def dls_step(q, target, lam=0.05):
    raise NotImplementedError  # TODO

lab.check("dls", dls_step, fk, jacobian)

## Checkpoint: Math
**M1 (Apply).** $\det J$ at $q_2 = 90°$?

**M2 (Remember).** Maximum reach in meters?

**M3 (Understand).** IK solutions for a point strictly inside the reachable annulus?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** With the arm fully stretched ($q_2 = 0$), which end-effector motion is impossible at that instant?
- a) motion perpendicular to the arm
- b) motion along the arm (radially)
- c) rotation of the whole arm about joint 1

**K2 (Evaluate).** Damped least squares has one knob, $\lambda$. Explain what too small and too large do, and how a
real arm controller picks it.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `p01.h`).
3. In `roboprep/checks/p01.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()